In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# --------------------------------------------------------------------------------
# 0. Stop any existing Spark session (important in notebooks)
# --------------------------------------------------------------------------------
try:
    spark.stop()
except:
    pass

# --------------------------------------------------------------------------------
# 1. Spark Session with MySQL Connector (auto-download via Maven)
# --------------------------------------------------------------------------------
spark = SparkSession.builder \
    .appName("scd1_mysql_integration") \
    .config("spark.jars.packages", "com.mysql:mysql-connector-j:8.0.33") \
    .getOrCreate()

# --------------------------------------------------------------------------------
# 2. MySQL Connection Parameters
# --------------------------------------------------------------------------------
HOST = "3.134.89.129"
USER = "root"
PASSWORD = "Yashwant!14"
DATABASE = "practice_db"   # ✅ updated DB name
TABLE = "target_employees"

jdbc_url = f"jdbc:mysql://{HOST}:3306/{DATABASE}?useSSL=false"

''' 
Initial Data for target_employees table:'
+--------+---------+--------+
| emp_id | name    | salary |
+--------+---------+--------+
|      1 | Yash    |  52000 |
|      2 | Mukesh  |  62000 |
|      3 | Ram     |  72000 |
|      4 | Krishna |  82000 |
+--------+---------+--------+
'''

schema = StructType([
    StructField("emp_id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("salary", IntegerType(), True)
])


initial_data = [
    (1, "Yash", 52000),
    (2, "Mukesh", 62000),
    (3, "Ram", 72000),
    (4, "Krishna", 82000), 
]

initial_data_df = spark.createDataFrame(initial_data, schema)


# --------------------------------------------------------------------------------
# 6. Write Back to MySQL (overwrite target table)
# --------------------------------------------------------------------------------
initial_data_df.write.format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", TABLE) \
    .option("truncate", "true") \
    .option("user", USER) \
    .option("password", PASSWORD) \
    .option("driver", "com.mysql.cj.jdbc.Driver") \
    .mode("overwrite") \
    .save()


# --------------------------------------------------------------------------------
# 3. Read Target Table from MySQL
# --------------------------------------------------------------------------------
data_target_df = spark.read.format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", TABLE) \
    .option("user", USER) \
    .option("password", PASSWORD) \
    .option("driver", "com.mysql.cj.jdbc.Driver") \
    .load()

print("Target Table (from MySQL):")
data_target_df.show()

# --------------------------------------------------------------------------------
# 4. New Incoming Data (simulate source feed)
# --------------------------------------------------------------------------------


data_new = [
    (1, "Yash", 52000),      # unchanged
    (2, "Mukesh", 60000),    # salary changed
    (3, "Charlie", 70000),   # name changed
    (5, "Radha", 90000),     # new record
]

data_new_df = spark.createDataFrame(data_new, schema)

print("New Data:")
data_new_df.show()

# --------------------------------------------------------------------------------
# 5. SCD1 Logic
# --------------------------------------------------------------------------------
# Changed records
changed_condition = ((data_new_df.name != data_target_df.name) |
                     (data_new_df.salary != data_target_df.salary))

changed_records_df = data_new_df.join(data_target_df, on="emp_id", how="inner") \
    .where(changed_condition) \
    .select(data_new_df.emp_id, data_new_df.name, data_new_df.salary)

print("Changed Records:")
changed_records_df.show()

# New records
new_records_df = data_new_df.join(data_target_df, on="emp_id", how="left_anti")
print("New Records:")
new_records_df.show()

# History records (removed in new feed)
history_records_df = data_target_df.join(data_new_df, on="emp_id", how="left_anti")
print("History Records:")
history_records_df.show()

# Unchanged records
unchanged_condition = ((data_new_df.name == data_target_df.name) &
                       (data_new_df.salary == data_target_df.salary))

unchanged_records_df = data_new_df.join(data_target_df, on="emp_id", how="inner") \
    .where(unchanged_condition) \
    .select(data_new_df.emp_id, data_new_df.name, data_new_df.salary)

print("Unchanged Records:")
unchanged_records_df.show()

# Final union
final_df = history_records_df.union(changed_records_df).union(new_records_df).union(unchanged_records_df).orderBy("emp_id")

print("Final DataFrame:")
final_df.show()

# --------------------------------------------------------------------------------
# 6. Write Back to MySQL (overwrite target table)
# --------------------------------------------------------------------------------
final_df.write.format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", TABLE) \
    .option("truncate", "true") \
    .option("user", USER) \
    .option("password", PASSWORD) \
    .option("driver", "com.mysql.cj.jdbc.Driver") \
    .mode("overwrite") \
    .save()

print("✅ Final DataFrame written back to MySQL target_employees")

# --------------------------------------------------------------------------------
# 6b. Re-read updated target table from MySQL (verification step)
# --------------------------------------------------------------------------------
updated_df = spark.read.format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", TABLE) \
    .option("user", USER) \
    .option("password", PASSWORD) \
    .option("driver", "com.mysql.cj.jdbc.Driver") \
    .load()

print("🔄 Updated Target Table (from MySQL after overwrite):")
updated_df.show()

# --------------------------------------------------------------------------------
# 7. Stop Spark
# --------------------------------------------------------------------------------
spark.stop()

:: loading settings :: url = jar:file:/Users/yash/Desktop/MyProjects/cloudprojects/dataeng/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/yash/.ivy2/cache
The jars for the packages stored in: /Users/yash/.ivy2/jars
com.mysql#mysql-connector-j added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-ccf55610-5803-43e8-8b0b-deb101a4d891;1.0
	confs: [default]
	found com.mysql#mysql-connector-j;8.0.33 in central
	found com.google.protobuf#protobuf-java;3.21.9 in central
:: resolution report :: resolve 54ms :: artifacts dl 2ms
	:: modules in use:
	com.google.protobuf#protobuf-java;3.21.9 from central in [default]
	com.mysql#mysql-connector-j;8.0.33 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   2   |   0   |   0   |   0   ||   2   |   0   |
	----------------------------------------

Target Table (from MySQL):


+------+-------+------+
|emp_id|   name|salary|
+------+-------+------+
|     4|Krishna| 82000|
|     3|    Ram| 72000|
|     1|   Yash| 52000|
|     2| Mukesh| 62000|
+------+-------+------+

New Data:
+------+-------+------+
|emp_id|   name|salary|
+------+-------+------+
|     1|   Yash| 52000|
|     2| Mukesh| 60000|
|     3|Charlie| 70000|
|     5|  Radha| 90000|
+------+-------+------+

Changed Records:


+------+-------+------+
|emp_id|   name|salary|
+------+-------+------+
|     2| Mukesh| 60000|
|     3|Charlie| 70000|
+------+-------+------+

New Records:


...

+------+-------+------+
|emp_id|   name|salary|
+------+-------+------+
|     4|Krishna| 82000|
+------+-------+------+

Unchanged Records:


+------+----+------+
|emp_id|name|salary|
+------+----+------+
|     1|Yash| 52000|
+------+----+------+

Final DataFrame:


+------+-------+------+
|emp_id|   name|salary|
+------+-------+------+
|     1|   Yash| 52000|
|     2| Mukesh| 60000|
|     3|Charlie| 70000|
|     4|Krishna| 82000|
|     5|  Radha| 90000|
+------+-------+------+



✅ Final DataFrame written back to MySQL target_employees
🔄 Updated Target Table (from MySQL after overwrite):


+------+-------+------+
|emp_id|   name|salary|
+------+-------+------+
|     1|   Yash| 52000|
|     2| Mukesh| 60000|
|     3|Charlie| 70000|
|     5|  Radha| 90000|
+------+-------+------+

